In [7]:
import os.path as op
import os
import numpy as np
import nibabel as nb
from ipyparallel import Client

TEST = 'EUT' #subjective value

In [ ]:
engines = Client()

In [5]:
#home_dir =  '/Users/mrenke/data/ds-stressrisk'
bids_folder = '/Users/mrenke/data/ds-stressrisk' #home_dir + '/data/'
script_folder='/Users/mrenke/git/stress_risk/stress_risk/fmri_analysis/glm_classic/fsl_fromGoki'

subFolders = [f for f in os.listdir(bids_folder) if f[0:3] == 'sub']

subs = subFolders #list(np.loadtxt(subs,str))


new_folder = op.join(bids_folder, 'derivatives', 'glm1st') 

sessions = [1, 2]
runs = map(str,range(1,7))

ignore_subs = []
subs = [x for x in subs if x not in ignore_subs]
print(subs)

['sub-13', 'sub-14', 'sub-22', 'sub-25', 'sub-49', 'sub-40', 'sub-47', 'sub-24', 'sub-23', 'sub-15', 'sub-12', 'sub-46', 'sub-41', 'sub-48', 'sub-52', 'sub-55', 'sub-30', 'sub-37', 'sub-01', 'sub-39', 'sub-54', 'sub-53', 'sub-38', 'sub-09', 'sub-36', 'sub-31', 'sub-44', 'sub-43', 'sub-17', 'sub-28', 'sub-10', 'sub-26', 'sub-19', 'sub-21', 'sub-42', 'sub-45', 'sub-20', 'sub-18', 'sub-16', 'sub-29', 'sub-34', 'sub-33', 'sub-05', 'sub-02', 'sub-56', 'sub-51', 'sub-58', 'sub-03', 'sub-04', 'sub-32', 'sub-35', 'sub-61', 'sub-59', 'sub-50', 'sub-57']


In [6]:
def make_new_dir(dir_name):
    if not op.exists(dir_name):
        os.mkdir(dir_name)

In [14]:
for sub in subs: #finds and replaces info in text file that writes analysis
    for ses in sessions:
        for run in runs:

            template = op.join(script_folder,'glm_1variable_step1.fsf')
            new_f = op.join(script_folder, 'glms_' + TEST + '_step1','_'.join([sub,sesh,run]) +'.fsf')
            
            make_new_dir(new_folder)
            
            with open(template,'r') as file:
                filedata = file.read()

            filedata = filedata.replace('$SUB',sub)
            filedata = filedata.replace('$RUN',run)
            filedata = filedata.replace('$SESSION',sesh)
            filedata = filedata.replace('$TEST',TEST)
            
            #next get number of time points
            func = op.join(bids_folder,'derivatives', 'fmriprep',sub,f'ses-{ses}', f'{sub}_ses-{ses}_task-risk_run-{run}_space-T1w_desc-preproc_bold.nii.gz')
            
            func = nb.load(func)
            ntpts = func.shape[-1]
            filedata = filedata.replace('$NUM_TIMEPOINTS',str(ntpts))

            
            with open(new_f,'w') as file:
                file.write(filedata)


In [15]:
def feat(in_tuple):
    sub,sesh,run = in_tuple
    fsf = op.join(new_folder,'_'.join([sub,sesh,run]) +'.fsf')
    
    cmd = ['feat',fsf]
    cmd = ' '.join(cmd)
    os.system(cmd)
    return in_tuple

In [16]:
in_tuples = []
for sub in subs:
    for sesh in sessions:
        for run in runs:
            in_tuples.append((sub,sesh,run))
        

In [17]:
sub_engines = engines[0:54] #less if you want to use fewer
sub_engines.push(dict(new_folder = new_folder))
sub_engines.execute('import os.path as op')
with sub_engines.sync_imports():
    import os
    
output = sub_engines.map_sync(feat,in_tuples)

importing os on engine(s)


# Now run fixed effects analyis across runs within each session

In [ ]:
print('finished')